# M5.0 · Negative result: CPU offload is not free

**Question:** Is overlap automatically a pipeline speedup? Recompute the original H2 negative result: slow PyTorch CPU encoding dominates even though CPU/GPU work overlaps for ~50 seconds. This is an expensive experiment; analyze evidence by default.

**Learning loop:** predict → inspect frozen evidence → compute → critique → propose a controlled experiment.

Reference lesson: [open course/M05_parallel_heterogeneous/LESSON.md](../course/M05_parallel_heterogeneous/LESSON.md). Switch language: [中文](./05_cpu_offload_negative.zh-CN.ipynb).

**Default mode:** CPU-only, internet-free, read-only. GPU work is never run implicitly.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## Inspect and recompute real evidence

Run cells in order. Explain every denominator and scope.

In [ ]:
h2 = ns.read_json("evidence/results/p5-H2-H2-pipeline/H2_RESULTS.json")
a = h2["A_sequential"]["total_s"]
b = h2["B_pipelined"]["total_s"]
overlap = h2["B_pipelined"]["overlap_s_total"]
print(f"Four requests: GPU-native sequential {a:.3f}s, CPU+GPU naive pipeline {b:.3f}s")
print(f"Real overlapping work {overlap:.3f}s, relative throughput A/B={a/b:.3f}x")
print(f"Pipeline is {b/a:.2f}x SLOWER (same four images), despite proven overlap.")
assert h2["B_pipelined"]["overlap_proven"] is True
plot_values = {"GPU native": a, "PyTorch CPU offload": b}
plot_ylabel, plot_title = "4-request wall seconds", "H2 negative result (NOT a new run)"


In [ ]:
SHOW_PLOT = False  # Optional: requires matplotlib in the notebook kernel
if SHOW_PLOT:
    import matplotlib.pyplot as plt
    labels = list(plot_values)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(labels, [plot_values[label] for label in labels])
    ax.set_ylabel(plot_ylabel)
    ax.set_title(plot_title)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Optional plot skipped; set SHOW_PLOT=True to visualize the measured values.")


## Your turn / Self-check

Explain the timeline bottleneck and identify which execution engine is limiting throughput. Propose a CPU encoder acceleration hypothesis that could recover this negative result.

**Evidence discipline:** Answer in your own words. This notebook's numbers come from existing committed records, not your GPU. If your optional smoke runs, its data must remain under `runs/` and may not be presented as a full-protocol median.
